In [ ]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
from scipy.stats import pearsonr
from dash import html, dcc, Input, Output, Dash
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths()

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability

In [ ]:
Logger().init_logger(None, None, logging_level="DEBUG")
animal_ids = [6]
paradigm = [1100]
session_range = [1,33]
session_ids = None
normalize = True
smooth = False
excl_session_names = ['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min']

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

In [ ]:
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)

In [ ]:
# firing rates and behavior data
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
fr_z_all_sess = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))

In [ ]:
# ensamble related data
ensambles = analytics.get_analytics('ConcatenatedEnsambles40ms', session_names=session_names)
ens_data = analytics.get_analytics('TrackwiseEnsembleProj', session_names=session_names)
ensamble_proj = analytics.get_analytics('ConcatenatedEnsambleProj40ms', session_names=session_names)#.drop("to_ephys_timestamp", axis=1)
ensamble_proj.set_index(['session_id'], inplace = True)
ens_data = ens_data.set_index(['session_id', 'trial_id']).sort_index()
Assembly = "Assembly007"

In [ ]:
#cue seperation plot 
# df = ens_data.reset_index() 
all_session_ids = ens_data.index.get_level_values("session_id").unique().tolist()
all_sessions_keep = list(range(1, len(all_session_ids) + 1))
# sessions_keep = [8, 9, 11, 12, 13, 14, 15, 16] # C1 strategy
# sessions_keep = [15, 16, 17, 18, 19, 20, 21, 22, 23] # C2 strategy
# sessions_keep = [23]
sessions_keep = all_sessions_keep # all sessions
print ("Sessions considered:", sessions_keep)

session_ids_keep = [all_session_ids[i - 1] for i in sessions_keep]

if len(sessions_keep) == len(all_sessions_keep):
    session_selection_label = "All Sessions"
elif len(sessions_keep) == 1:
    session_selection_label = f"Session {sessions_keep[0]}"
else:
    session_selection_label = f"Sessions {min(sessions_keep)} to {max(sessions_keep)}"

# keep only expert trials (C1 cue -> choice_R1 = True, choice_R2 = False; C2 cue -> choice_R1 = False, choice_R2 = True)
ens_filt = ens_data.loc[ens_data.index.get_level_values("session_id").isin(session_ids_keep)]

expert1 = ens_filt[ # Keep only expert trials
        (ens_filt["choice_R1"] == 1) &
        (ens_filt["choice_R2"] == 0) &
        (ens_filt["cue"] == 1)
    ]

expert2 = ens_filt[ # Keep only expert trials
        (ens_filt["choice_R1"] == 0) &
        (ens_filt["choice_R2"] == 1) &
        (ens_filt["cue"] == 2)
    ]

# expert1 = ens_filt[ # Keep only non expert trials
#         (ens_filt["choice_R1"] == 0) &
#         (ens_filt["choice_R2"] == 1) &
#         (ens_filt["cue"] == 1)
#     ]

# expert2 = ens_filt[ # Keep only non expert trials
#         (ens_filt["choice_R2"] == 0) &
#         (ens_filt["choice_R1"] == 1) &
#         (ens_filt["cue"] == 2)
#     ]



ens_filt = pd.concat([expert1, expert2]).copy()


cutoff_date = pd.Timestamp("2025-01-17")
session_dates = pd.to_datetime(ens_filt.index.get_level_values("session_id").astype(str).str[:10])
is_before_reversal = session_dates <= cutoff_date
is_after_reversal = session_dates > cutoff_date

# Raw `cue` labels are reward-zone-sticky after reversal; `visual_cue` follows the actual cue identity.
ens_filt["visual_cue"] = ens_filt["cue"]
ens_filt.loc[is_after_reversal & ens_filt["cue"].eq(1).to_numpy(), "visual_cue"] = 2
ens_filt.loc[is_after_reversal & ens_filt["cue"].eq(2).to_numpy(), "visual_cue"] = 1

ens_before = ens_filt.loc[is_before_reversal]
ens_after = ens_filt.loc[is_after_reversal]

def average_by_cue(ens_subset):
    trial_avg = (
        ens_subset # .reset_index()
        .groupby(["from_position_bin", "visual_cue"], as_index=False)[Assembly]
        .mean()
        .rename(columns={Assembly: "Assembly_mean", "visual_cue": "cue"})
    )
    trial_avg = trial_avg[trial_avg["cue"].isin([1, 2])]
    trial_avg["from_position_bin"] = trial_avg["from_position_bin"].astype(str)
    return trial_avg

def count_trials_by_cue(ens_subset):
    trial_counts = (
        ens_subset.reset_index()[["session_id", "trial_id", "visual_cue"]]
        .loc[lambda df: df["visual_cue"].isin([1, 2])]
        .drop_duplicates(["session_id", "trial_id", "visual_cue"])
        .groupby("visual_cue")
        .size()
    )
    return {cue: int(trial_counts.get(cue, 0)) for cue in [1, 2]}

trial_avg_before = average_by_cue(ens_before)
trial_avg_after = average_by_cue(ens_after)
trial_avg_all = average_by_cue(ens_filt)
trial_counts_before = count_trials_by_cue(ens_before)
trial_counts_after = count_trials_by_cue(ens_after)
trial_counts_all = count_trials_by_cue(ens_filt)
trial_avg = pd.concat([trial_avg_before, trial_avg_after, trial_avg_all], ignore_index=True)

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{"colspan": 2}, None], [{}, {}]],
    subplot_titles=(
        f"{session_selection_label} overall",
        "Before Reversal (<= 2025-01-17)",
        "After Reversal (> 2025-01-17)",
    ),
    shared_yaxes=True,
    vertical_spacing=0.12,
    horizontal_spacing=0.04,
)

plot_specs = [
    (1, 1, trial_avg_all, trial_counts_all),
    (2, 1, trial_avg_before, trial_counts_before),
    (2, 2, trial_avg_after, trial_counts_after),
]

for row, col, trial_avg_split, trial_counts in plot_specs:
    cue1 = trial_avg_split[trial_avg_split["cue"] == 1]
    cue2 = trial_avg_split[trial_avg_split["cue"] == 2]
    cue_colors = {1: "orange", 2: "purple"}

    fig.add_trace(
        go.Scatter(
            x=cue1["from_position_bin"],
            y=cue1["Assembly_mean"],
            mode="lines",
            name="cue 1",
            line=dict(color=cue_colors[1], width=3),
            opacity=0.7,
            legendgroup="cue 1",
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    fig.add_trace(
        go.Scatter(
            x=cue2["from_position_bin"],
            y=cue2["Assembly_mean"],
            mode="lines",
            name="cue 2",
            line=dict(color=cue_colors[2], width=3),
            opacity=0.7,
            legendgroup="cue 2",
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    subplot = fig.get_subplot(row, col)
    fig.add_annotation(
        x=subplot.xaxis.domain[1] - 0.01,
        y=subplot.yaxis.domain[1] - 0.02,
        xref="paper",
        yref="paper",
        text=(
            f"<span style='color:{cue_colors[1]}'>cue 1 (n={trial_counts[1]})</span><br>"
            f"<span style='color:{cue_colors[2]}'>cue 2 (n={trial_counts[2]})</span>"
        ),
        showarrow=False,
        xanchor="right",
        yanchor="top",
        align="left",
        bgcolor="rgba(255,255,255,0.75)",
        bordercolor="rgba(0,0,0,0.15)",
        borderwidth=1,
        font=dict(size=10),
    )

fig.update_layout(
    title = f"Average Ensemble Activity trackwise - {session_selection_label} (Expert Only)",
    yaxis_title=f"Avg({Assembly}) per position bin",
    width=1200,
    height=567,
    showlegend=False,
)

for row, col, _, _ in plot_specs:
    fig.update_xaxes(title_text="from_position_bin", type="linear", row=row, col=col)
fig.update_yaxes(title_text=f"Avg({Assembly}) per position bin", row=2, col=1)
fig.update_yaxes(range=[min(trial_avg["Assembly_mean"]), 1], title_font=dict(size=11), tickfont=dict(size=10))

for row, col, _, _ in plot_specs:
    fig.add_vrect(x0=-80, x1=25,  fillcolor="orange", opacity=0.1, layer="below", line_width=0, row=row, col=col)
    fig.add_vrect(x0=50,  x1=110, fillcolor="grey",   opacity=0.1, layer="below", line_width=0, row=row, col=col)
    fig.add_vrect(x0=170, x1=230, fillcolor="grey",   opacity=0.1, layer="below", line_width=0, row=row, col=col)
    fig.add_vline(x=-120, line_dash="dash", line_color="black", line_width=1, row=row, col=col)


fig.show()



In [ ]:
# Average ensemble separated by choice_R1 vs choice_R2
ens_choice = ens_data.loc[ens_data.index.get_level_values("session_id").isin(session_ids_keep)]

choice_R1 = (
    ens_choice
    .groupby(["from_position_bin", "choice_R1"], as_index=False)[Assembly]
    .mean()
    .rename(columns={Assembly: "Assembly_mean"})
)
choice_R2 = (
    ens_choice
    .groupby(["from_position_bin", "choice_R2"], as_index=False)[Assembly]
    .mean()
    .rename(columns={Assembly: "Assembly_mean"})
)
choice_R1_only = (
    ens_choice[(ens_choice["choice_R1"] == 1) & (ens_choice["choice_R2"] == 0)]
    .groupby("from_position_bin", as_index=False)[Assembly]
    .mean()
    .rename(columns={Assembly: "Assembly_mean"})
)
choice_R2_only = (
    ens_choice[(ens_choice["choice_R2"] == 1) & (ens_choice["choice_R1"] == 0)]
    .groupby("from_position_bin", as_index=False)[Assembly]
    .mean()
    .rename(columns={Assembly: "Assembly_mean"})
)
choice_R1["from_position_bin"] = choice_R1["from_position_bin"].astype(str)
choice_R2["from_position_bin"] = choice_R2["from_position_bin"].astype(str)
choice_R1_only["from_position_bin"] = choice_R1_only["from_position_bin"].astype(str)
choice_R2_only["from_position_bin"] = choice_R2_only["from_position_bin"].astype(str)

fig = make_subplots(rows=1, cols=3, subplot_titles=("choice_R1", "choice_R2", "R1/R2 choice only"), shared_yaxes=True)
for value, label, color, opacity in [(1, "stop", "#0072B2", 0.8), (0, "skip", "#B8860B", 0.9)]:
    choice_R1_value = choice_R1[choice_R1["choice_R1"] == value]
    fig.add_trace(go.Scatter(x=choice_R1_value["from_position_bin"], y=choice_R1_value["Assembly_mean"], mode="lines", name=label, line=dict(color=color, width=3), opacity=opacity, legend="legend"), row=1, col=1)
for value, label, color, opacity in [(1, "stop", "#0072B2", 0.8), (0, "skip", "#B8860B", 0.9)]:
    choice_R2_value = choice_R2[choice_R2["choice_R2"] == value]
    fig.add_trace(go.Scatter(x=choice_R2_value["from_position_bin"], y=choice_R2_value["Assembly_mean"], mode="lines", name=label, line=dict(color=color, width=3), opacity=opacity, legend="legend2"), row=1, col=2)
fig.add_trace(go.Scatter(x=choice_R1_only["from_position_bin"], y=choice_R1_only["Assembly_mean"], mode="lines", name="R1 only", line=dict(color="#0072B2", width=3), opacity=0.8, legend="legend3"), row=1, col=3)
fig.add_trace(go.Scatter(x=choice_R2_only["from_position_bin"], y=choice_R2_only["Assembly_mean"], mode="lines", name="R2 only", line=dict(color="#B8860B", width=3), opacity=0.9, legend="legend3"), row=1, col=3)
y_min = pd.concat([choice_R1["Assembly_mean"], choice_R2["Assembly_mean"], choice_R1_only["Assembly_mean"], choice_R2_only["Assembly_mean"]], ignore_index=True).min()

fig.update_layout(
    title=f"Average Ensemble Activity trackwise by choice - {session_selection_label}",
    yaxis_title=f"Avg({Assembly}) per position bin",
    width=1400,
    legend=dict(x=fig.layout.xaxis.domain[0] + 0.01, y=fig.layout.yaxis.domain[1] - 0.02, xanchor="left", yanchor="top", bgcolor="rgba(255,255,255,0.7)"),
    legend2=dict(x=fig.layout.xaxis2.domain[0] + 0.01, y=fig.layout.yaxis2.domain[1] - 0.02, xanchor="left", yanchor="top", bgcolor="rgba(255,255,255,0.7)"),
    legend3=dict(x=fig.layout.xaxis3.domain[0] + 0.01, y=fig.layout.yaxis.domain[1] - 0.02, xanchor="left", yanchor="top", bgcolor="rgba(255,255,255,0.7)"),
)
fig.update_xaxes(title_text="from_position_bin", type="linear", row=1, col=1)
fig.update_xaxes(title_text="from_position_bin", type="linear", row=1, col=2)
fig.update_xaxes(title_text="from_position_bin", type="linear", row=1, col=3)
fig.update_yaxes(range=[y_min, 1])

for col in [1, 2, 3]:
    fig.add_vrect(x0=-80, x1=25, fillcolor="orange", opacity=0.1, layer="below", line_width=0, row=1, col=col)
    fig.add_vrect(x0=50, x1=110, fillcolor="grey", opacity=0.1, layer="below", line_width=0, row=1, col=col)
    fig.add_vrect(x0=170, x1=230, fillcolor="grey", opacity=0.1, layer="below", line_width=0, row=1, col=col)
    fig.add_vline(x=-120, line_dash="dash", line_color="black", line_width=1, row=1, col=col)

fig.show()

In [ ]:
# ordering neurons by ensemble weight (canonical Unit#### labels)
assembly_name = Assembly


def _index_to_unit_label(idx):
    if isinstance(idx, str):
        s = idx.strip()
        if s.startswith("Unit"):
            digits = "".join(ch for ch in s if ch.isdigit())
            if digits:
                return f"Unit{int(digits):04d}"
        if s.isdigit():
            return f"Unit{int(s) + 1:04d}"
    if isinstance(idx, (int, np.integer)):
        return f"Unit{int(idx) + 1:04d}"
    try:
        return f"Unit{int(idx) + 1:04d}"
    except Exception:
        return None


def build_weight_order(ens_df, assembly_col):
    weights_raw = ens_df[assembly_col].copy()
    unit_labels = [_index_to_unit_label(idx) for idx in weights_raw.index]
    weights_by_unit = pd.Series(weights_raw.to_numpy(), index=unit_labels, name=assembly_col)
    weights_by_unit = weights_by_unit[weights_by_unit.index.notna()].groupby(level=0).mean()

    order_df = (
        pd.DataFrame({"unit": weights_by_unit.index, "weight": weights_by_unit.values})
        .assign(abs_weight=lambda d: d["weight"].abs())
        .sort_values(["abs_weight", "unit"], ascending=[True, True], kind="mergesort")
    )
    ordered_units = order_df["unit"].tolist()
    return ordered_units, weights_by_unit.reindex(ordered_units)


neurons_ordered, weights_by_unit = build_weight_order(ensambles, assembly_name)
neurons_ordered


In [ ]:
meta_data = {}
meta_data['SpikeClusterMetadata'] = analytics.get_analytics('SpikeClusterMetadata', mode='set',
                                                      #  columns = cols,
                                                       paradigm_ids=paradigm,
                                                       animal_ids=animal_ids,
                                                       excl_session_names=excl_session_names,
                                                       session_ids=session_ids)


meta_data['SpikeClusterMetadata']

In [ ]:
meta_data = {}
meta_data['SpikeClusterMetadata'] = analytics.get_analytics('SpikeClusterMetadata', mode='set',
                                                      #  columns = cols,
                                                       paradigm_ids=paradigm,
                                                       animal_ids=animal_ids,
                                                       excl_session_names=excl_session_names,
                                                       session_ids=session_ids)

# # HPC
# meta_data['SpikeClusterMetadata'] = meta_data['SpikeClusterMetadata'][meta_data['SpikeClusterMetadata'].cluster_id<=20]
# # mPFC
# meta_data['SpikeClusterMetadata'] = meta_data['SpikeClusterMetadata'][meta_data['SpikeClusterMetadata'].cluster_id<20]
# meta_data['SpikeClusterMetadata']

In [ ]:
neurons_ordered


In [ ]:
i = Assembly
spike_metadata = meta_data["SpikeClusterMetadata"]
fig_heat = plot_unit_fr_stability.render_plot_heatmap(spike_metadata)
heatmap = fig_heat.data[0]

raw_heatmap_labels = np.array(fig_heat.layout.yaxis.ticktext, dtype=object)
cluster_to_unit = {}
if "cluster_id_str" in spike_metadata.columns:
    cluster_to_unit = (
        spike_metadata[["cluster_id", "cluster_id_str"]]
        .dropna()
        .drop_duplicates("cluster_id")
        .assign(cluster_id=lambda d: d["cluster_id"].astype(str))
        .set_index("cluster_id")["cluster_id_str"]
        .to_dict()
    )


def _heatmap_label_to_unit_label(label):
    s = str(label).strip()
    if s in cluster_to_unit:
        return str(cluster_to_unit[s])
    if s.startswith("Unit"):
        digits = "".join(ch for ch in s if ch.isdigit())
        if digits:
            return f"Unit{int(digits):04d}"
    if s.isdigit():
        return f"Unit{int(s):04d}"
    return s


current_labels = np.array(
    [_heatmap_label_to_unit_label(label) for label in raw_heatmap_labels],
    dtype=str,
)
current_label_set = set(current_labels)

desired_order = [lab for lab in neurons_ordered if lab in current_label_set]
if not desired_order:
    raise RuntimeError(
        "No overlap between ensemble unit labels and firing-rate heatmap rows. "
        f"Example heatmap labels: {raw_heatmap_labels[:5].tolist()}; "
        f"example ensemble labels: {neurons_ordered[:5]}"
    )

label_to_row = {lab: j for j, lab in enumerate(current_labels)}
row_indices = [label_to_row[lab] for lab in desired_order]

z = np.array(heatmap.z)
z_reordered = z[row_indices, :]
xlabels = np.array(fig_heat.layout.xaxis.ticktext, dtype=str)

# subplot creation
fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.3, 0.7],
    subplot_titles=(i, "Average Firing Rate per Neuron/Session"),
)

# Left plot
if i == assembly_name:
    weights_full = weights_by_unit.copy()
else:
    _, weights_full = build_weight_order(ensambles, i)
weights = weights_full.reindex(desired_order).dropna()

# HPC mPFC split by unit id
labels_hp_ordered = [lab for lab in desired_order if int(lab.replace("Unit", "")) <= 20 and lab in weights.index]
labels_mpfc_ordered = [lab for lab in desired_order if int(lab.replace("Unit", "")) > 20 and lab in weights.index]

w_hp = weights.loc[labels_hp_ordered].to_numpy()
w_mpfc = weights.loc[labels_mpfc_ordered].to_numpy()


def make_stem_lines(x_vals, y_labels, color):
    """Horizontal 'stems' at categorical y positions."""
    xs, ys = [], []
    for x, ylab in zip(x_vals, y_labels):
        xs.extend([0, x, None])
        ys.extend([ylab, ylab, None])
    return go.Scatter(
        x=xs,
        y=ys,
        mode="lines",
        line=dict(color=color),
        showlegend=False,
        hoverinfo="skip",
    )


hp_stems = make_stem_lines(w_hp, labels_hp_ordered, color="blue")
mpfc_stems = make_stem_lines(w_mpfc, labels_mpfc_ordered, color="magenta")

hp_markers = go.Scatter(
    x=w_hp,
    y=labels_hp_ordered,
    mode="markers",
    name=f"{i}, HP",
    marker=dict(color="blue", symbol="circle"),
)

mpfc_markers = go.Scatter(
    x=w_mpfc,
    y=labels_mpfc_ordered,
    mode="markers",
    name=f"{i}, mPFC",
    marker=dict(color="magenta", symbol="circle"),
)

thr = 0.2
mask = np.abs(weights.values) > thr
x_high = weights.values[mask]
y_high = weights.index.values[mask]

highlight = go.Scatter(
    x=x_high,
    y=y_high,
    mode="markers",
    name="Otsu thresh.",
    marker=dict(
        size=10,
        color="rgba(0,0,0,0)",
        line=dict(color="black", width=2),
        symbol="circle",
    ),
)

fig.add_trace(hp_stems, row=1, col=1)
fig.add_trace(mpfc_stems, row=1, col=1)
fig.add_trace(hp_markers, row=1, col=1)
fig.add_trace(mpfc_markers, row=1, col=1)
fig.add_trace(highlight, row=1, col=1)

fig.update_xaxes(
    title_text="Weight",
    range=[-0.45, 0.45],
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Neurons",
    categoryorder="array",
    categoryarray=desired_order,
    dtick=1,
    row=1,
    col=1,
)

fig.add_vline(
    x=0,
    line_width=1,
    line_color="black",
    row=1,
    col=1,
)

# Heatmap
heatmap_trace = go.Heatmap(
    z=z_reordered,
    x=xlabels,
    y=desired_order,
    colorscale=heatmap.colorscale,
    zmin=heatmap.zmin,
    zmax=heatmap.zmax,
    colorbar=dict(
        title=heatmap.colorbar.title.text,
        tickvals=heatmap.colorbar.tickvals,
        ticktext=heatmap.colorbar.ticktext,
        tickmode=heatmap.colorbar.tickmode,
        len=heatmap.colorbar.len,
        thickness=heatmap.colorbar.thickness,
        tickfont=dict(size=heatmap.colorbar.tickfont.size),
    ),
)

fig.add_trace(heatmap_trace, row=1, col=2)

fig.update_xaxes(
    title_text=fig_heat.layout.xaxis.title.text,
    row=1,
    col=2,
)

fig.update_yaxes(
    title_text=fig_heat.layout.yaxis.title.text,
    categoryorder="array",
    dtick=1,
    categoryarray=desired_order,
    row=1,
    col=2,
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=800,
    legend=dict(font=dict(size=6)),
)

fig.show()


In [ ]:
# recreated altered heatmap, wider 
spike_metadata = meta_data["SpikeClusterMetadata"]
fig_heat = plot_unit_fr_stability.render_plot_heatmap(spike_metadata)
heatmap = fig_heat.data[0]

z = np.array(heatmap.z, dtype=float)

raw_heatmap_labels = np.array(fig_heat.layout.yaxis.ticktext, dtype=object)
if raw_heatmap_labels.size != z.shape[0]:
    raise RuntimeError(
        "Heatmap row labels do not match heatmap rows. "
        f"Got {raw_heatmap_labels.size} labels for {z.shape[0]} rows."
    )

cluster_to_unit = {}
if "cluster_id_str" in spike_metadata.columns:
    cluster_to_unit = (
        spike_metadata[["cluster_id", "cluster_id_str"]]
        .dropna()
        .drop_duplicates("cluster_id")
        .assign(cluster_id=lambda d: d["cluster_id"].astype(str))
        .set_index("cluster_id")["cluster_id_str"]
        .to_dict()
    )


def _heatmap_label_to_unit_label(label):
    s = str(label).strip()
    if s in cluster_to_unit:
        return str(cluster_to_unit[s])
    if s.startswith("Unit"):
        digits = "".join(ch for ch in s if ch.isdigit())
        if digits:
            return f"Unit{int(digits):04d}"
    if s.isdigit():
        return f"Unit{int(s):04d}"
    return s


current_labels = np.array(
    [_heatmap_label_to_unit_label(label) for label in raw_heatmap_labels],
    dtype=str,
)

mean_fr = np.nanmean(z, axis=1)
order_base = pd.DataFrame(
    {
        "unit": current_labels,
        "mean_fr": mean_fr,
        "row_index": np.arange(z.shape[0]),
    }
)


def _unit_number(unit_label):
    digits = "".join(ch for ch in str(unit_label) if ch.isdigit())
    return int(digits) if digits else np.inf


def _unit_label_color(unit_label):
    # yellow = HP (units 1-20), magenta = mPFC (units 21+), matching experimental_ensembles colorway
    return "#C47900" if _unit_number(unit_label) <= 20 else "#FF00FF"


def _display_unit_label(unit_label):
    unit_number = _unit_number(unit_label)
    return str(unit_number) if np.isfinite(unit_number) else str(unit_label)


def _sparse_tick_positions(n_rows, max_labels=16):
    if n_rows == 0:
        return np.array([], dtype=int)
    step = max(1, int(np.ceil(n_rows / max_labels)))
    ticks = np.arange(0, n_rows, step)
    if ticks[-1] != n_rows - 1:
        ticks = np.append(ticks, n_rows - 1)
    return ticks


def _subplot_axis_ref(axis_name, col, domain=False):
    axis_ref = axis_name if col == 1 else f"{axis_name}{col}"
    return f"{axis_ref} domain" if domain else axis_ref


fr_order = (
    order_base
    .assign(mean_fr_sort=lambda d: d["mean_fr"].fillna(-np.inf))
    .sort_values(["mean_fr_sort", "unit"], ascending=[False, True], kind="mergesort")
)

unit_id_order = (
    order_base
    .assign(unit_number=lambda d: d["unit"].map(_unit_number))
    .sort_values(["unit_number", "unit"], ascending=[True, True], kind="mergesort")
)

raw_xlabels = np.array(fig_heat.layout.xaxis.ticktext, dtype=object)
if raw_xlabels.size != z.shape[1]:
    raw_xlabels = np.array(heatmap.x, dtype=object)
xlabels_date = np.array([str(label).split("_")[0] for label in raw_xlabels], dtype=str)

# x-position of the boundary after the last 2025-01-17 column
jan17_cols = np.where(xlabels_date == "2025-01-17")[0]
jan17_boundary = float(jan17_cols[-1]) + 0.5 if len(jan17_cols) > 0 else None

x_positions = np.arange(z.shape[1])

# One tick per unique date, placed at the date's first column — avoids label overlap
_first_occ = np.sort(np.unique(xlabels_date, return_index=True)[1])
x_tickvals = x_positions[_first_occ]
x_ticktext = xlabels_date[_first_occ]

orders = [
    ("Sorted by mean firing rate", fr_order),
    ("Sorted by neuron ID", unit_id_order),
]

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Average firing rate per neuron/session", "Average firing rate per neuron/session"],
    horizontal_spacing=0.12,
)
for annotation in fig.layout.annotations:
    annotation.font = dict(family="Arial", size=16, color="black")

for col, (_, order_df) in enumerate(orders, start=1):
    row_indices = order_df["row_index"].to_numpy()
    ordered_labels = order_df["unit"].tolist()
    z_ordered = z[row_indices, :]
    y_positions = np.arange(z_ordered.shape[0])
    customdata = np.empty((*z_ordered.shape, 2), dtype=object)
    customdata[:, :, 0] = np.array(ordered_labels, dtype=object)[:, None]
    customdata[:, :, 1] = xlabels_date[None, :]

    heatmap_kwargs = dict(
        z=z_ordered,
        x=x_positions,
        y=y_positions,
        customdata=customdata,
        colorscale=heatmap.colorscale,
        zmin=heatmap.zmin,
        zmax=heatmap.zmax,
        showscale=(col == 2),
        hovertemplate=(
            "Neuron: %{customdata[0]}<br>"
            "Date: %{customdata[1]}<br>"
            "Avg firing rate: %{z:.3g} Hz<extra></extra>"
        ),
    )
    if col == 2:
        heatmap_kwargs["colorbar"] = dict(
            title=heatmap.colorbar.title.text,
            tickvals=heatmap.colorbar.tickvals,
            ticktext=heatmap.colorbar.ticktext,
            tickmode=heatmap.colorbar.tickmode,
            len=heatmap.colorbar.len,
            thickness=heatmap.colorbar.thickness,
            tickfont=dict(family="Arial", size=12, color="black"),
        )

    fig.add_trace(
        go.Heatmap(**heatmap_kwargs),
        row=1,
        col=col,
    )

    for xpos in np.flatnonzero(xlabels_date[1:] != xlabels_date[:-1]) + 0.5:
        fig.add_vline(
            x=xpos,
            line=dict(color="black", width=1.5, dash="dot"),
            opacity=0.6,
            row=1,
            col=col,
        )

    if jan17_boundary is not None:
        fig.add_vline(
            x=jan17_boundary,
            line=dict(color="#215CAF", width=2),
            row=1,
            col=col,
        )

    fig.update_xaxes(
        title_text="Session",
        tickmode="array",
        tickvals=x_tickvals,
        ticktext=x_ticktext,
        tickangle=-45,
        tickfont=dict(family="Arial", size=12, color="black"),
        title_font=dict(family="Arial", size=14, color="black"),
        ticks="outside",
        ticklen=4,
        showline=True,
        linewidth=1,
        linecolor="black",
        automargin=True,
        row=1,
        col=col,
    )

    y_tickvals = _sparse_tick_positions(len(ordered_labels), max_labels=16)
    # Always include the boundary rows (units 20 and 21) to make the color split visible
    priority_positions = np.array(
        [i for i, lbl in enumerate(ordered_labels) if _unit_number(lbl) in (20, 21)],
        dtype=int,
    )
    y_tickvals = np.unique(np.concatenate([y_tickvals, priority_positions])).astype(int)
    y_ticklabels = [ordered_labels[pos] for pos in y_tickvals]

    fig.update_yaxes(
        title_text="Neuron ID",
        tickmode="array",
        tickvals=y_tickvals,
        ticktext=[""] * len(y_tickvals),
        title_font=dict(family="Arial", size=14, color="black"),
        ticks="outside",
        ticklen=4,
        showline=True,
        linewidth=1,
        linecolor="black",
        autorange="reversed",
        row=1,
        col=col,
    )

    for y_tick, label in zip(y_tickvals, y_ticklabels):
        fig.add_annotation(
            x=-0.01,
            y=int(y_tick),
            xref=_subplot_axis_ref("x", col, domain=True),
            yref=_subplot_axis_ref("y", col),
            text=_display_unit_label(label),
            showarrow=False,
            xanchor="right",
            yanchor="middle",
            align="right",
            font=dict(family="Arial", size=12, color=_unit_label_color(label)),
        )

fig.update_layout(
    title=dict(
        text="Average firing rate per neuron/session",
        x=0.5,
        xanchor="center",
        font=dict(family="Arial", size=18, color="black"),
    ),
    font=dict(family="Arial", size=12, color="black"),
    template="plotly_white",
    width=1800,
    height=600,
    margin=dict(l=120, r=130, t=70, b=130),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

In [ ]:
session_used = '2024-12-03_16-23'
neuron_data_s15 = fr.xs(session_used, level=2)

# Filter Unit columns only
unit_cols = [col for col in neuron_data_s15.columns if isinstance(col, str) and col.startswith('Unit')]
neuron_data_s15 = neuron_data_s15[unit_cols]

# Compute correlation matrix (neuron vs neuron)
corr_matrix = neuron_data_s15.corr()

# Create ordering based on reversed neurons_ordered
reversed_order = neurons_ordered[::-1]
unit_order = [u for u in reversed_order if u in corr_matrix.columns]

# Reorder the correlation matrix
corr_matrix_ordered = corr_matrix.loc[unit_order, unit_order]

# Create heatmap
fig = px.imshow(
    corr_matrix_ordered,
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    labels=dict(x="Neurons", y="Neurons", color="Pearson r"),
    title=f"Neuron-Neuron Pearson Correlation Matrix (Session {session_used}, {Assembly})"
)

fig.update_xaxes(tickangle=90, tickfont=dict(size=6))
fig.update_yaxes(tickfont=dict(size=6))
fig.update_layout(width=800, height=800)

fig.show()


In [ ]:
# selecting only the rows of fr_track where column from_position_bin  is between 148 and 151
#fr_track_selected = fr_track[(fr_track['from_position_bin'] >= (-95)) & (fr_track['from_position_bin'] <= 10)] # cue zone
# fr_track_selected = fr_track[(fr_track['from_position_bin'] >= (50)) & (fr_track['from_position_bin'] <= 110)] # R1 zone
fr_track_selected = fr_track[(fr_track['from_position_bin'] >= (170)) & (fr_track['from_position_bin'] <= 230)] #R2 zone

# further filter to rows where cue == 1
# fr_track_selected = fr_track_selected[fr_track_selected['cue'] == 1]
# fr_track_selected = fr_track_selected[fr_track_selected['choice_R2'] == False]

In [ ]:
session_ids_all = fr_track_selected.index.get_level_values("session_id").unique()
session_ids_all = pd.to_datetime(session_ids_all, format="%Y-%m-%d_%H-%M")
session_ids_all

In [ ]:
start = pd.Timestamp("2024-11-25")
end   = pd.Timestamp("2025-01-26")

# set top x neurons and select sessions
top_x_neurons = neurons_ordered[-20:]

session_ids_all = fr_track_selected.index.get_level_values("session_id").unique()
dt_index = pd.to_datetime(session_ids_all, format="%Y-%m-%d_%H-%M")

# boolean selection
mask = (dt_index >= start) & (dt_index <= end)
selected = session_ids_all[mask]



n_sessions = len(selected)

n_cols = 4 # int(np.ceil(np.sqrt(n_sessions)))
n_rows = 5 # int(np.ceil(n_sessions / n_cols))

reversed_order_topx = top_x_neurons[::-1]

# Create subplots
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    #subplot_titles=[f'S{s}' for s in selected],
    horizontal_spacing=0.02,
    vertical_spacing=0.03,
)

# Compute correlation matrix for each session (only top 10 neurons)
for idx, session_id in enumerate(selected):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    
    try:
        neuron_data_session = fr_track_selected.xs(session_id, level=2)
        
        # Filter to Unit columns only
        unit_cols = [c for c in neuron_data_session.columns if isinstance(c, str) and c.startswith('Unit')]
        neuron_data_session = neuron_data_session[unit_cols]
        
        # Select top neurons in canonical weight order
        topx_unit_cols = [u for u in top_x_neurons if u in neuron_data_session.columns]
        
        # Filter to only top neurons
        neuron_data_top10 = neuron_data_session[topx_unit_cols]
        
        # Compute correlation matrix
        corr_matrix = neuron_data_top10.corr()
        
        # Create ordering based on reversed top neurons
        unit_order = [u for u in reversed_order_topx if u in corr_matrix.columns]
        unit_labels = [str(int(u.replace('Unit', ''))) for u in unit_order]
        
        # Reorder the correlation matrix
        corr_matrix_ordered = corr_matrix.loc[unit_order, unit_order]
        
        # Add heatmap to subplot
        fig.add_trace(
            go.Heatmap(
                z=corr_matrix_ordered.values,
                x=unit_labels,
                y=unit_labels,
                colorscale='RdBu_r',
                zmin=-1,
                zmax=1,
                showscale=(idx == 0),
                colorbar=dict(
                    title="Pearson r",
                    x=1.02
                ) if idx == 0 else None,
                hovertemplate='X: %{x}<br>Y: %{y}<br>r: %{z:.3f}<extra></extra>'
            ),
            row=row,
            col=col
        )

        fig.add_annotation(
            text=f"S{session_id.split('_')[0]}",
            xref=f"x{idx + 1}",
            yref=f"y{idx + 1}",
            x=7,
            y=19.5,
            xanchor="left",
            yanchor="top",
            showarrow=False,
            font=dict(size=10, color="black"),
            bgcolor="rgba(255,255,255,0.2)"
        )
        
        # Update axes - show tick labels for top neurons
        fig.update_xaxes(showticklabels=True, tickangle=90, tickfont=dict(size=8), row=row, col=col)
        fig.update_yaxes(showticklabels=True, tickfont=dict(size=8), row=row, col=col)
        
    except Exception as e:
        print(f"Error processing session {session_id}: {e}")
        continue

# Update layout
fig.update_layout(
    title_text=f"Top 20 Most Weighted Neurons: Pearson Correlation Matrices Across Sessions {selected[0].split('_')[0]} to {selected[-1].split('_')[0]}<br>({Assembly} ordered by weights)",
    title_font_size=16,
    width=300 * n_cols,
    height=300 * n_rows,
    showlegend=False,
    #title=dict(y=0.9),
)


fig.show()

print(f"Created {n_sessions} correlation heatmaps (10x10) in {n_rows}x{n_cols} grid")
print(f"Top 20 most weighted neurons: {top_x_neurons}")


In [ ]:
top_ns = [5, 10, 20, 30, 50]
session_ids_all = fr_track_selected.index.get_level_values("session_id").unique()
fr_used = fr_track_selected.copy()

def mean_offdiag_corr(corr_df: pd.DataFrame) -> float:
    k = corr_df.shape[0]
    if k < 2:
        return np.nan
    iu = np.triu_indices(k, k=1)
    vals = corr_df.to_numpy()[iu]
    return float(np.nanmean(vals))

# second y-axis for ensemble mean
fig = make_subplots(specs=[[{"secondary_y": True}]])

for top_n in top_ns:
    top_n_neurons = neurons_ordered[-top_n:]
    rows = []

    for session_id in session_ids_all:
        try:
            neuron_data_session = fr_used.xs(session_id, level=2)

            unit_cols = [
                c for c in neuron_data_session.columns
                if isinstance(c, str) and c.startswith("Unit")
            ]
            neuron_data_session = neuron_data_session[unit_cols]

            top_unit_cols = [u for u in top_n_neurons if u in neuron_data_session.columns]

            neuron_data_top = neuron_data_session[top_unit_cols]
            print(neuron_data_top.keys())
            corr = neuron_data_top.corr(method="pearson")
            avg_corr = mean_offdiag_corr(corr)

            rows.append((session_id, avg_corr))
        except Exception:
            rows.append((session_id, np.nan))

    df_avg = pd.DataFrame(rows, columns=["session_id", "avg_pairwise_corr"])

    fig.add_trace(
        go.Scatter(
            x=df_avg["session_id"],
            y=df_avg["avg_pairwise_corr"],
            mode="lines+markers",
            name=f"top {top_n} neuron correlation",
        ),
        secondary_y=False,
    )

# ensemble mean per session (one trace on the right y-axis)
ens_mean = (
    ensamble_proj[Assembly]
    .groupby(level=0)
    .mean()
    .reindex(session_ids_all)
)

fig.add_trace(
    go.Scatter(
        x=ens_mean.index,
        y=ens_mean.values,
        mode="lines+markers",
        name=f"Ensemble mean ({Assembly})",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Average pairwise Pearson correlation across sessions",
    width=900,
    height=450,
)

fig.update_xaxes(title_text="Session")

fig.update_yaxes(
    title_text="Average pairwise correlation (Pearson r)",
    range=[-1, 1],
    secondary_y=False,
)

fig.update_yaxes(
    title_text="Ensemble activation (mean)",
    secondary_y=True,
)

fig.show()


In [ ]:
top_ns = [10, 20, 77]
fr_used = fr_z_all_sess.copy()

# 2x2 layout: correlations on first row, firing rates on second row
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Pearson corr (unweighted)", "Pearson corr (weighted)",
        "Mean firing rate (unweighted)", "Mean firing rate (weighted)"
    ),
    shared_yaxes=False,
)

weights = weights_by_unit.copy()

# pearson correlations for single neurons weighted and unweightes
for top_n in top_ns:
    top_n_neurons = neurons_ordered[-top_n:]
    rows_unw, rows_w = [], []

    for session_id in session_ids_all:
        neuron_data_session = fr_used.xs(session_id, level=2)

        unit_cols = [
            c for c in neuron_data_session.columns
            if isinstance(c, str) and c.startswith("Unit")
        ]
        neuron_data_session = neuron_data_session[unit_cols]

        top_unit_cols = [u for u in top_n_neurons if u in neuron_data_session.columns]

        neuron_data_top = neuron_data_session[top_unit_cols]

        corr_unw = neuron_data_top.corr(method="pearson")
        rows_unw.append((session_id, mean_offdiag_corr(corr_unw)))

        w = weights.reindex(top_unit_cols)
        neuron_data_top_w = neuron_data_top.mul(w, axis=1)
        corr_w = neuron_data_top_w.corr(method="pearson")
        rows_w.append((session_id, mean_offdiag_corr(corr_w)))

    df_unw = pd.DataFrame(rows_unw, columns=["session_id", "avg_corr"])
    df_w = pd.DataFrame(rows_w, columns=["session_id", "avg_corr"])

    fig.add_trace(
        go.Scatter(
            x=df_unw["session_id"],
            y=df_unw["avg_corr"],
            mode="lines+markers",
            name=f"top {top_n}",
            legendgroup=f"top{top_n}",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=df_w["session_id"],
            y=df_w["avg_corr"],
            mode="lines+markers",
            name=f"top {top_n}",
            legendgroup=f"top{top_n}",
            showlegend=False,  # avoid duplicated legend entries
        ),
        row=1,
        col=2,
    )


# mean fr
fr_units = fr_used.loc[:, fr_used.columns.str.startswith("Unit")]
# fr_units = fr.loc[:, fr.columns.str.startswith("Unit")]
fr_units = fr_units[fr_units.index.get_level_values(2) != 10]

w_all = weights_by_unit.reindex(fr_units.columns).fillna(0.0)
fr_units_weighted = fr_units.mul(w_all, axis=1)

session_means_unweighted = (
    fr_units
    .groupby(level=2)
    .mean()
    .mean(axis=1)
)

session_means_weighted = (
    fr_units_weighted
    .groupby(level=2)
    .mean()
    .mean(axis=1)
)

fig.add_scatter(
    x=session_means_unweighted.index,
    y=session_means_unweighted.values,
    mode="lines+markers",
    name="Mean FR unweighted",
    row=2,
    col=1,
)

fig.add_scatter(
    x=session_means_weighted.index,
    y=session_means_weighted.values,
    mode="lines+markers",
    name="Mean FR weighted",
    row=2,
    col=2,
)


fig.update_layout(
    title="Session-wise Pearson correlations (top row) and mean firing rates (bottom row)",
    width=1200,
    height=800,
)

fig.update_xaxes(title_text="Session", row=1, col=1)
fig.update_xaxes(title_text="Session", row=1, col=2)
fig.update_xaxes(title_text="Session", row=2, col=1)
fig.update_xaxes(title_text="Session", row=2, col=2)

fig.update_yaxes(title_text="Avg pairwise Pearson r", row=1, col=1)
fig.update_yaxes(title_text="Avg pairwise Pearson r", row=1, col=2)
fig.update_yaxes(title_text="Mean firing rate (z-scored)", row=2, col=1)
fig.update_yaxes(title_text="Mean firing rate (z-scored)", row=2, col=2)

fig.update_yaxes(range=[-1, 1], row=1, col=1)
fig.update_yaxes(range=[-1, 1], row=1, col=2)

fig.show()


In [ ]:
s = '2024-12-09_17-45'
neuron_data_s15 = fr_track.xs(s, level=2)[fr_track.xs(s, level=2)['trial_id'] == 23]
unit_cols = [col for col in neuron_data_s15.columns if isinstance(col, str) and col.startswith('Unit')]
neuron_data_s15 = neuron_data_s15[unit_cols]

corr_matrix = neuron_data_s15.corr()

reversed_order = neurons_ordered[::-1]
unit_order = [u for u in reversed_order if u in corr_matrix.columns]

corr_matrix_ordered = corr_matrix.loc[unit_order, unit_order]

fig = px.imshow(
    corr_matrix_ordered,
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    labels=dict(x="Neurons", y="Neurons", color="Pearson r"),
    title=f"Neuron-Neuron Pearson Correlation Matrix (Session {s}, {Assembly} ordering reversed)"
)

fig.update_xaxes(tickangle=90, tickfont=dict(size=6))
fig.update_yaxes(tickfont=dict(size=6))
fig.update_layout(width=900, height=900)

fig.show()


In [ ]:
# selectiung only the data of ens_data in the cue zone and right before the reward zones (-95 to 10; 148-151; 168-171)
ens_data_selected = ens_data[(ens_data['from_position_bin'] >= (-95)) & (ens_data['from_position_bin'] <= 10)]
# adding reward zones with bins 148-151; 168-171
ens_data_selected = pd.concat([
    ens_data_selected,
    ens_data[(ens_data['from_position_bin'] >= 148) & (ens_data['from_position_bin'] <= 151)],
    ens_data[(ens_data['from_position_bin'] >= 168) & (ens_data['from_position_bin'] <= 171)]
])

In [ ]:
from_to_session =  ("2024-12-03", "2024-12-11")

df = ens_data

# If session_id is in the index, bring it back as a column
if "session_id" not in df.columns:
    if isinstance(df.index, pd.MultiIndex) and "session_id" in df.index.names:
        df = df.reset_index()
    elif df.index.name == "session_id":
        df = df.reset_index()

trial_avg = (
    df.groupby(["session_id", "from_position_bin", "cue"], as_index=False)[Assembly]
      .mean()
      .rename(columns={Assembly: "Assembly_mean"})
)

trial_avg = trial_avg[trial_avg["cue"].isin([1, 2])].copy()

if from_to_session is not None:
    start, end = map(pd.Timestamp, from_to_session)
    session_dates = pd.to_datetime(
        trial_avg["session_id"].astype(str).str[:10],
        format="%Y-%m-%d",
        errors="coerce",
    )
    trial_avg = trial_avg.loc[(session_dates >= start) & (session_dates <= end)].copy()

if trial_avg.empty:
    raise ValueError("No sessions available for the selected from_to_session range.")

# sorting x-axis values
pos_numeric = pd.to_numeric(trial_avg["from_position_bin"], errors="coerce")
if pos_numeric.notna().any():
    trial_avg["_pos"] = pos_numeric
    x_order = np.sort(trial_avg["_pos"].dropna().unique())
    # keep a numeric x for correct ordering/spacing
    trial_avg["_x"] = trial_avg["_pos"]
    x_ticks = x_order
    x_ticktext = [str(int(x)) if float(x).is_integer() else str(x) for x in x_order]
else:
    # fallback: treat as ordered categorical by appearance
    trial_avg["_x"] = trial_avg["from_position_bin"].astype(str)
    x_order = list(pd.unique(trial_avg["_x"]))
    x_ticks = x_order
    x_ticktext = x_order

# Ridgeline plot parameters
sessions = sorted(trial_avg["session_id"].unique(), reverse=True)
dy = 0.8          # vertical spacing between sessions
cue_offset = 0.02 # vertical separation between cue 1 and cue 2 within a session
amp = 0.2

fig = go.Figure()

for i, s in enumerate(sessions):
    base = i * dy

    for cue, off in [(1, -cue_offset), (2, +cue_offset)]:
        sub = trial_avg[(trial_avg["session_id"] == s) & (trial_avg["cue"] == cue)].copy()

        # Ensure every x bin exists, fill missing with 0 (or np.nan if you prefer gaps)
        sub = sub.set_index("_x").reindex(x_order).reset_index()

        y = sub["Assembly_mean"].fillna(0.0).to_numpy()
        y_ridge = base + off + amp * y

        fig.add_trace(
            go.Scatter(
                x=sub["_x"],
                y=y_ridge,
                mode="lines",
                line=dict(width=1, color= cue == 1 and "orange" or "purple"),
                fill= None,
                name=f"session {s} | cue {cue}",
                hovertemplate=(
                    "session=%{customdata[0]}<br>"
                    "cue=%{customdata[1]}<br>"
                    "from_position_bin=%{x}<br>"
                    f"mean({Assembly})=%{{customdata[2]:.4f}}<extra></extra>"
                ),
                customdata=np.c_[np.full(len(sub), s), np.full(len(sub), cue), y],
                showlegend=True,
            )
        )



#Layout polish
fig.update_layout(
    title=f"Ridgeline: mean({Assembly}) per from_position_bin, one ridge per session",
    xaxis_title="from_position_bin",
    yaxis_title="session_id (stacked ridges)",
    # hovermode="x",
    height=max(450, 40 * len(sessions) + 200),
)

# Put session labels on the y-axis at each session baseline
fig.update_yaxes(
    tickmode="array",
    tickvals=[i * dy for i in range(len(sessions))],
    ticktext=[str(s)[:10] for s in sessions],
)

fig.add_vrect(x0=-80, x1=25,  fillcolor="orange", opacity=0.2, layer="below", line_width=0)
fig.add_vrect(x0=50,  x1=110, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)
fig.add_vrect(x0=170, x1=230, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)
fig.add_vline(x=-120, line_dash="dash", line_color="black", line_width=1)


if pos_numeric.notna().any():
    fig.update_xaxes(
        tickmode="array",
        ticktext=x_ticktext,
        showgrid=False, zeroline=False
    )

fig.show()

In [ ]:
# trial based ridgline
df = ens_data
need_reset = False
if isinstance(df.index, pd.MultiIndex):
    need_reset = any(name in df.index.names for name in ["session_id", "trial_id"])
elif df.index.name in ["session_id", "trial_id"]:
    need_reset = True

if need_reset:
    df = df.reset_index()

# session selector
# session_to_plot = df["session_id"].dropna().unique()[0]  # first available session as default
session_to_plot = '2025-01-25_21-29'

df_s = df.loc[df["session_id"] == session_to_plot].copy()
#df_s = df_s[df_s["cue"].isin([1, 2])].copy()

# If "cue" is constant per trial, this will keep it; if not, it will take the first observed cue per trial
trial_cue = (
    df_s.groupby("trial_id", as_index=True)["cue"]
        .agg(lambda x: x.dropna().iloc[0] if len(x.dropna()) else np.nan)
)

# Aggregate within trial and position bin
trial_avg = (
    df_s.groupby(["trial_id", "from_position_bin"], as_index=False)[Assembly]
        .mean()
        .rename(columns={Assembly: "Assembly_mean"})
)


# Build ordered numeric x if possible
pos_numeric = pd.to_numeric(trial_avg["from_position_bin"], errors="coerce")
if pos_numeric.notna().any():
    trial_avg["_x"] = pos_numeric
    x_order = np.sort(trial_avg["_x"].dropna().unique())
    x_tickvals = x_order
    x_ticktext = [str(int(x)) if float(x).is_integer() else str(x) for x in x_order]
else:
    trial_avg["_x"] = trial_avg["from_position_bin"].astype(str)
    x_order = list(pd.unique(trial_avg["_x"]))
    x_tickvals = x_order
    x_ticktext = x_order


# Ridgeline parameters
trials = sorted(trial_avg["trial_id"].unique())
dy = 0.8      #between trials
amp = 0.05    # amplitude scaling

fig = go.Figure()

for i, t in enumerate(trials):
    base = i * dy

    sub = trial_avg.loc[trial_avg["trial_id"] == t, ["_x", "Assembly_mean"]].copy()
    sub = sub.set_index("_x").reindex(x_order).reset_index()

    y = sub["Assembly_mean"].fillna(0.0).to_numpy()
    y_ridge = base + amp * y

    cue_t = trial_cue.get(t, np.nan)
    line_color = "orange" if cue_t == 1 else "purple"

    fig.add_trace(
        go.Scatter(
            x=sub["_x"],
            y=y_ridge,
            mode="lines",
            line=dict(width=1, color=line_color),
            name=f"trial {t}",
            hovertemplate=(
                "session=%{customdata[0]}<br>"
                "trial_id=%{customdata[1]}<br>"
                "cue=%{customdata[2]}<br>"
                "from_position_bin=%{x}<br>"
                f"mean({Assembly})=%{{customdata[3]:.4f}}<extra></extra>"
            ),
            customdata=np.c_[
                np.full(len(sub), session_to_plot),
                np.full(len(sub), t),
                np.full(len(sub), cue_t),
                y,
            ],
            showlegend=False,
        )
    )

# Y-axis
fig.update_yaxes(
    tickmode="array",
    tickvals=[i * dy for i in range(len(trials))],
    ticktext=[str(t) for t in trials],
    title_text="trial_id",
)

fig.update_layout(
    title=f"Ridgeline: mean({Assembly}) per from_position_bin, one ridge per trial (session {session_to_plot})",
    xaxis_title="from_position_bin",
    height=max(500, 30 * len(trials) + 250),
)

fig.update_xaxes(
    tickmode="array",
    showgrid=False,
    zeroline=False,
)

fig.add_vrect(x0=-80, x1=25,  fillcolor="orange", opacity=0.2, layer="below", line_width=0)
fig.add_vrect(x0=50,  x1=110, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)
fig.add_vrect(x0=170, x1=230, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)
fig.add_vline(x=-120, line_dash="dash", line_color="black", line_width=1)

fig.show()


In [ ]:
i = '2024-12-06_16-49'
s1 = ens_data.xs(i, level='session_id')
s1_avg = (s1.groupby(['from_position_bin','cue'], as_index=False)[Assembly].mean())
title = f'Average Ensemble Activityby Position Across Trials in Session {i}'
fig = px.line(
    s1_avg,
    x='from_position_bin',
    y=Assembly,
    color='cue',
    color_discrete_map={1: 'orange', 2: 'purple'},
    markers=True,
    title=title,
    range_y=[-1, 1.5]
)
fig.add_vrect(x0=-80, x1=25,  fillcolor="orange", opacity=0.3, layer="below", line_width=0)
fig.add_vrect(x0=50,  x1=110, fillcolor="grey",   opacity=0.4, layer="below", line_width=0)
fig.add_vrect(x0=170, x1=230, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)
fig.add_vline(x=-120, line_dash="dash", line_color="black", line_width=1)

fig.show()


# plotting the avg. activity for each trial in session 14 in a scatter plot
s1 = ens_data_selected.xs(i, level='session_id')
s1_avg = s1.groupby('trial_id').mean().reset_index()
s1_avg['trial_id'] = s1_avg['trial_id'] - 1
title = f'Average Ensemble Activity in the Cue Zones and Before Reward Zones by Trial for Session {i}'
# fig = px.line(s1_avg, x='trial_id', y=Assembly, title=title, range_y=[-1,2])
fig = px.line(
    s1_avg,
    x='trial_id',
    y=Assembly,
    color='cue',
    color_discrete_map={1: 'orange', 2: 'purple'},
    markers=True,
    title=title,
    range_y=[-1, 2]
)
fig.show()

In [ ]:
s = '2024-12-06_16-49' #session
i = 19 #trial
s1_trial1 = ens_data.xs((s, i), level=('session_id', 'trial_id'))
title = f'Firing Rate by Position for Trial {i} in Session {s}'

fig = px.line(s1_trial1, x='from_position_bin', y=Assembly, color = 'cue', title=title)
fig.add_vrect(x0=-80, x1=25, fillcolor="orange", opacity=0.3, layer="below", line_width=0)
fig.add_vrect(x0=50, x1=110, fillcolor="grey", opacity=0.4, layer="below", line_width=0)
fig.add_vrect(x0=170, x1=230, fillcolor="grey", opacity=0.2, layer="below", line_width=0)
fig.add_vline(x=-120, line_dash="dash", line_color="black", line_width=1)
fig.show()

print(f'Trial {i-1} is a Cue {s1_trial1["cue"]} trial in Session {s}')

# Distance Score according to Sun et al. (2025)

- D = |A_near − A_far|/max(A_near, A_far)
- Pearson correlation

to compute differences between near and far trials over sessions for: 
- Top n neurons of ensemble
- All neurons weighted
- Ensmeble

In [ ]:
# dropping all neurons that are not in top n
n = 20
top_x_neurons = neurons_ordered[-n:]
top_x_neurons
top_unit_cols = [u for u in top_x_neurons if u in fr_track.columns]
mask = (~fr_track.columns.str.startswith("Unit")) | (fr_track.columns.isin(top_unit_cols))


mask
fr_top_n = fr_track.loc[:, mask]
fr_top_n


In [ ]:
# distance scoire function following sun et al
def get_distance_score(A_near, A_far):
    denom = np.maximum(A_near, A_far)
    out = np.zeros_like(denom, dtype=float)
    m = denom != 0
    out[m] = np.abs(A_near[m] - A_far[m]) / denom[m]
    return out

distance_scores_cue = []
distance_scores_r1 = []
distance_scores_r2  = []

for session_id in fr_track.index.get_level_values('session_id').unique():
    drop_session_data = fr_track.xs(session_id, level='session_id')

    a_near_cue = []
    a_near_r1 = []
    a_near_r2 = []

    a_far_cue = []
    a_far_r1 = []
    a_far_r2 = []

    # max for each trial and append to near or far depending on the cue
    for trial_id in drop_session_data['trial_id'].values.unique():
        trial_data = drop_session_data[drop_session_data['trial_id'] == trial_id]  # [Assembly]
        mask_cols = trial_data.columns.str.startswith("Unit")
        a_trial_cue = trial_data.loc[trial_data['from_position_bin'].between(-80, 25), mask_cols].max().max()
        a_trial_r1 = trial_data.loc[trial_data['from_position_bin'].between(50, 110), mask_cols].max().max()
        a_trial_r2 = trial_data.loc[trial_data['from_position_bin'].between(170, 230), mask_cols].max().max()
        if (trial_data['cue'] == 1).all():
            a_near_cue.append(a_trial_cue)
            a_near_r1.append(a_trial_r1)
            a_near_r2.append(a_trial_r2)
        elif (trial_data['cue'] == 2).all():
            a_far_cue.append(a_trial_cue)
            a_far_r1.append(a_trial_r1)
            a_far_r2.append(a_trial_r2)
        else:
            print(f"Warning: trial {trial_id} in session {session_id} has mixed cues. Skipping.")
    distance_score_cue = get_distance_score(np.mean(a_near_cue), np.mean(a_far_cue))
    distance_score_r1 = get_distance_score(np.mean(a_near_r1), np.mean(a_far_r1))
    distance_score_r2 = get_distance_score(np.mean(a_near_r2), np.mean(a_far_r2))


    distance_scores_cue.append({'session_id': session_id, 'distance_score': distance_score_cue})
    distance_scores_r1.append({'session_id': session_id, 'distance_score': distance_score_r1})
    distance_scores_r2.append({'session_id': session_id, 'distance_score': distance_score_r2})


distance_scores_cue

In [ ]:
def get_distance_score(A_near, A_far):
    denom = np.maximum(A_near, A_far)
    out = np.zeros_like(denom, dtype=float)
    m = denom != 0
    out[m] = np.abs(A_near[m] - A_far[m]) / denom[m]
    return out

windows = {
    'start-end': (-169, -80),
    "cue": (-80, 25),
    # "r1":  (50, 110),
    # "r2":  (170, 230),
    'r1+r2': (25, 230),
}

unit_prefix = "Unit"
results = []

for session_id in fr_top_n.index.get_level_values("session_id").unique():
    drop_session_data = fr_top_n.xs(session_id, level="session_id")
    unit_cols = drop_session_data.columns[drop_session_data.columns.str.startswith(unit_prefix)]

    near = {k: [] for k in windows}
    far  = {k: [] for k in windows}

    for trial_id in drop_session_data["trial_id"].unique():
        trial_data = drop_session_data[drop_session_data["trial_id"] == trial_id]

        if (trial_data["cue"] == 1).all():
            group = near
        elif (trial_data["cue"] == 2).all():
            group = far
        else:
            continue

        for wname, (lo, hi) in windows.items():
            s = (
                trial_data
                .loc[trial_data["from_position_bin"].between(lo, hi), unit_cols]
                .mean(axis=0)   # mean activation per unit
            )
            group[wname].append(s)

    for wname in windows:
        n1 = len(near[wname])
        n2 = len(far[wname])
        n_pair = min(n1, n2)

        #  distance scores according to Sun et al. 
        if n1 == 0 or n2 == 0:
            dist = pd.Series(np.nan, index=unit_cols)
        else:
            A_near = pd.concat(near[wname], axis=1).mean(axis=1)
            A_far  = pd.concat(far[wname],  axis=1).mean(axis=1)
            dist = pd.Series(
                get_distance_score(A_near.to_numpy(), A_far.to_numpy()),
                index=unit_cols
            )

        # Pearson correlaation
        if n_pair < 2:
            pearson_r = pd.Series(np.nan, index=unit_cols)
            pearson_p = pd.Series(np.nan, index=unit_cols)
        else:
            X = pd.concat(near[wname][:n_pair], axis=1)
            Y = pd.concat(far[wname][:n_pair],  axis=1)

            r_vals = np.empty(len(unit_cols))
            p_vals = np.empty(len(unit_cols))

            for i, u in enumerate(unit_cols):
                x = X.loc[u].to_numpy()
                y = Y.loc[u].to_numpy()

                if np.allclose(x, x[0]) or np.allclose(y, y[0]):
                    r_vals[i] = np.nan
                    p_vals[i] = np.nan
                else:
                    r_vals[i], p_vals[i] = pearsonr(x, y)

            pearson_r = pd.Series(r_vals, index=unit_cols)
            pearson_p = pd.Series(p_vals, index=unit_cols)

        tmp = pd.DataFrame({
            "unit": unit_cols,
            "distance_score": dist.values,
            "pearson_r": pearson_r.values,
            "pearson_p": pearson_p.values,
            "session_id": session_id,
            "window": wname,
            "n_trials_cue1": n1,
            "n_trials_cue2": n2,
            "n_pairs_used": n_pair,
        })

        results.append(tmp)

distance_metrics_per_unit = (
    pd.concat(results, ignore_index=True)
      .set_index(["session_id", "window", "unit"])
      .sort_index()
)

distance_metrics_per_unit


In [ ]:
fr_track